# Atlas GasNet: predicción del consumo de gas

Prueba de concepto de Inteligencia Artificial Avanzada, UTN FRBA.
Este cuaderno usa exclusivamente Python y datos públicos ya reunidos.
No requiere ejecutar el extractor ni cargar archivos manualmente.

Se ajustan las transformaciones y los estimadores solo con entrenamiento.
La validación selecciona la época. Las dos pruebas finales se ejecutan después
de esa selección y no se utilizan para modificar el modelo.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = "https://github.com/brianUtn98/atlas-gasnet.git"
CODE_REVISION = "7bb86975198e448322453c932867593ffe7f4561"
ROOT = Path("/content/atlas-gasnet")
if not (ROOT / ".git").exists():
    subprocess.run(["git", "clone", "--branch", "work", REPOSITORY, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", CODE_REVISION], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "training/requirements.txt")], check=True)
print("Código:", subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
# Un proceso nuevo evita usar versiones de NumPy cargadas antes de la instalación.
def run_python(script):
    result = subprocess.run([sys.executable, "-c", script], cwd=ROOT,
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
    result.check_returncode()

run_python('from pathlib import Path\nfrom zipfile import ZipFile\nfrom training.bundles import prepare, load_prepared\nroot = Path.cwd()\nsource = root / "data/extractions/poc-v3"\nif not source.exists():\n    with ZipFile(root / "data/colab/poc-v3.zip") as archive:\n        archive.extractall(source)\nprepared = root / "models/poc-prepared"\nif prepared.exists():\n    load_prepared(prepared)\nelse:\n    prepare(source, prepared, "poc")\nprint("Datos preparados y verificados.")\n')


In [ ]:
run_python('from pathlib import Path\nfrom training.bundles import train_bundle, load_run\nroot = Path.cwd()\nrun = root / "models/poc-run"\nif run.exists():\n    load_run(run)\n    print("Se conserva el modelo ya entrenado; no se sobrescribe.")\nelse:\n    train_bundle(root / "models/poc-prepared", run)\nprint("Modelo listo.")\n')


In [ ]:
run_python('import json\nfrom pathlib import Path\nfrom training.evaluation import evaluate\nroot = Path.cwd()\nrun = root / "models/poc-run"\nevaluate(run, root / "models/poc-prepared", verification=(run / "evaluation").exists())\nreport = json.loads((run / "evaluation/metrics.json").read_text())\nfor name, section in [("validation", report["validation_diagnostics"]), *report["final_tests"].items()]:\n    print(name)\n    for estimator, summary in section["estimators"].items():\n        print(estimator, summary["aggregate"])\n')


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

history = json.loads((ROOT / "models/poc-run/history.json").read_text())
pd.DataFrame(history["epochs"]).plot(x="epoch", y=["train", "validation"])
plt.xlabel("Época")
plt.ylabel("Error cuadrático medio en la escala transformada")
plt.show()
print("Época seleccionada:", history["best_epoch"])

evaluation = ROOT / "models/poc-run/evaluation"
for split in ("validation", "test", "unseen-contract"):
    print(split)
    frame = pd.read_csv(evaluation / f"{split}.csv")
    frame["error_absoluto"] = frame["mlp_residual"].abs()
    display(frame.sort_values("error_absoluto").head(3))
    display(frame.sort_values("error_absoluto", ascending=False).head(3))
    display(Image(filename=str(evaluation / f"{split}.png")))


In [ ]:
# Guardar el modelo y las evaluaciones antes de cerrar la sesión.
import shutil
from google.colab import files

archive = shutil.make_archive("/content/atlas-gasnet-resultados", "zip", ROOT / "models/poc-run")
files.download(archive)


## Resultados conservados de la ejecución del 21/10/2026

El entrenamiento de Colab terminó en 61 épocas y seleccionó la época 36.
Su MAE de validación fue de 2.006,23 gas. Las pruebas finales, ejecutadas
después de seleccionar el modelo, dieron estos resultados:

| Conjunto | Estimador | Casos | MAE (gas) | RMSE (gas) | R² |
| --- | --- | ---: | ---: | ---: | ---: |
| Prueba temporal | MLP | 2.466 | 1.740,20 | 3.811,34 | 0,8629 |
| Prueba temporal | Mediana por función | 2.466 | 6.821,99 | 9.908,33 | 0,0732 |
| Prueba con USDT | MLP | 4.903 | 3.395,39 | 4.405,60 | 0,7906 |
| Prueba con USDT | Mediana por función | 4.903 | 8.604,64 | 11.554,49 | −0,4401 |

[Resultados completos, predicciones y pesos conservados en GitHub](https://github.com/brianUtn98/atlas-gasnet/tree/main/results/colab-2026-10-21).
Una nueva ejecución puede producir pequeñas diferencias numéricas según el entorno.
Los resultados anteriores corresponden al modelo conservado, sin ajustes posteriores
basados en las pruebas finales. La evaluación considera pocos contratos y un período
corto: no demuestra igual precisión para cualquier contrato o período de Ethereum.
